# A tour of the tiny ExeaLM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exea-labs/ExeaLM/blob/main/notebooks/tiny_tour.ipynb)

For new contributors. You need no GPU and no experience training models: **Runtime → Run all** works on Colab's
default CPU runtime. In testing on one CPU thread, the whole notebook took under 5 minutes. You will:

1. install ExeaLM;
2. train the `tiny` model on made-up data;
3. look at the circuits inside one TONN layer;
4. switch a group of circuits off and watch the loss rise (this is what TONN's search measures);
5. train two agents that can only answer questions by talking over the message board.

Background: the [README](https://github.com/exea-labs/ExeaLM#readme),
[docs/architecture.md](https://github.com/exea-labs/ExeaLM/blob/main/docs/architecture.md) and
[docs/swarms.md](https://github.com/exea-labs/ExeaLM/blob/main/docs/swarms.md).

## 1. Install

On Colab this clones the repository. If you opened the notebook from your own clone, it uses that instead.

In [ ]:
import os
import subprocess
import sys

if os.path.isdir("../exealm"):                      # opened from a clone's notebooks/ folder
    os.chdir("..")
elif not os.path.isdir("exealm"):                   # Colab, or anywhere else: get the code
    if not os.path.isdir("ExeaLM"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/exea-labs/ExeaLM"], check=True)
    os.chdir("ExeaLM")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-e", "."], check=True)
sys.path.insert(0, os.getcwd())                     # import from this folder, no runtime restart needed
print("working in", os.getcwd())

In [ ]:
import contextlib
import io
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

import exealm

print("torch", torch.__version__, "| exealm from", os.path.dirname(exealm.__file__))
DEVICE = "cpu"          # the CPU even on a GPU runtime: the models are tiny, and this is what the notebook was tested on
START = time.time()

## 2. Train the tiny model

The `tiny` preset has 2 TONN layers of 8 circuits each, about 127,000 parameters. It trains on made-up data: documents
from a random Markov chain, where each of 256 tokens can be followed by one of 4 tokens. Before training the model
guesses among 256 tokens (8 bits per token). A model that has learned the chain needs about 2 bits per token, the
chain's own randomness.

While it trains, two things happen that a normal transformer doesn't do:
- every batch is split into **swarms** of 2 agents that share a message board (the AG circuit);
- every 100 steps in the first half of training, TONN's **search** measures what each kind of circuit is worth and
  replaces the useless ones.

This is the same command as `python -m exealm.train.pretrain --preset tiny --data synthetic`. It takes 1 to 3
minutes.

In [ ]:
from exealm.train.pretrain import main as pretrain

OUT = "runs/tour"
if os.path.exists(f"{OUT}/log.jsonl"):
    os.remove(f"{OUT}/log.jsonl")                   # the log is appended to: start clean if you run this cell again
t0 = time.time()
with contextlib.redirect_stdout(io.StringIO()):     # pretrain prints one JSON line per event; we read its log instead
    model = pretrain(["--preset", "tiny", "--data", "synthetic", "--steps", "1500", "--eval_every", "100",
                      "--device", DEVICE, "--out", OUT])
log = [json.loads(line) for line in open(f"{OUT}/log.jsonl")]
evals = [r for r in log if r["event"] == "eval"]
print(f"trained {log[0]['params']:,} parameters in {time.time() - t0:.0f} s")
print(f"validation: {evals[0]['val_bits_per_token']} bits per token at step {evals[0]['step']}, "
      f"{evals[-1]['val_bits_per_token']} at step {evals[-1]['step']}")

In [ ]:
steps = [r["step"] for r in evals]
plt.figure(figsize=(6, 3))
plt.plot(steps, [r["val_bits_per_token"] for r in evals], marker="o", markersize=4)
plt.axhline(2.0, color="gray", linestyle="--", linewidth=1)
plt.text(steps[0], 1.9, "the chain's own randomness, about 2 bits", fontsize=8, color="gray", va="center")
plt.ylim(1.8, None)
plt.xlabel("training step")
plt.ylabel("bits per token")
plt.title("Validation loss of the tiny model")
plt.show()

## 3. Look inside one TONN layer

A TONN layer reads the residual stream into a row of **slots**. Each slot holds one small circuit: a delay, a short
convolution, a running sum, a small attention head, and so on (14 types). Every layer keeps at least one **RG**
(retrieval from a library) and at least one **AG** (the agents' message board). We trained without a library, so RG outputs
zeros here.

Nobody chose these circuits by hand. They were drawn at random, and the search may have swapped some.

In [ ]:
from exealm.model.circuits import CIRCUITS

layer = model.layers[0]
print(f"layer 0: {layer.K} slots of {layer.h} channels\n")
for k, spec in enumerate(layer.specs):
    t = spec["tmpl"]
    lag = f"lag {spec['lag']}" if CIRCUITS[t][1] else ""
    print(f"slot {k}  {t:<4}{lag:<8}{CIRCUITS[t][3]}")
print("\nboth layers, slots of each type:")
for i, row in enumerate(model.architecture()):
    print(f"  layer {i}: {row}")

In [ ]:
print("What the search did (worth = how much the loss rises without that type, bits per token per slot):")
for r in log:
    if r["event"] == "search":
        best = ", ".join(f"{t} {w:+.3f}" for t, w in list(r["type_worth"].items())[:3])
        print(f"  step {r['step']:4d}: replaced {r['replaced']} slot(s); most useful: {best}")

## 4. Switch circuits off and watch the loss rise

Every slot's output is multiplied by `layer.mask[k]`: 1 means on, 0 means off. Setting a group of slots to 0 removes
those circuits without touching any weights. How much the loss rises is what the group is worth.

TONN's search does exactly this (`group_worth` in `exealm/model/search.py`). It switches off one type in one layer at
a time, and replaces circuits worth nothing or less. With synthetic data there is no separate held-out split, so we
measure on fixed windows of the same made-up corpus, as the training script does.

In [ ]:
from exealm import MemoryContext
from exealm.data.loader import SyntheticShards

data, rng, swarm = SyntheticShards(), np.random.default_rng(123), model.cfg.swarm_size
batches = []
for _ in range(8):
    x, y, docs = data.batch(8, 128, rng, swarm, "mixed")
    batches.append((x, y, MemoryContext(doc_ids=docs, swarm_size=swarm)))


@torch.no_grad()
def bits_per_token():
    model.eval()
    return sum(float(model(x, m, y)[1]) for x, y, m in batches) / len(batches) / np.log(2)


counts = model.architecture()[0]
t = max((c for c in counts if c not in ("RG", "AG")), key=counts.get)    # layer 0's biggest group
ks = [k for k, s in enumerate(layer.specs) if s["tmpl"] == t]
base = bits_per_token()
layer.mask[ks] = 0.0                                 # switch the group off
off = bits_per_token()
layer.mask[ks] = 1.0                                 # and back on
again = bits_per_token()
for label, bits in (("everything on", base), (f"layer 0's {t} slots {ks} off", off), ("switched back on", again)):
    print(f"{label + ':':<34}{bits:.3f} bits per token")
print(f"\nswitching them off costs {off - base:+.3f} bits per token")

In [ ]:
from exealm.model.search import group_worth

W, per_type, _ = group_worth(model, batches)       # every (layer, type) group, the way the search measures it
rows = []
for i, lay in enumerate(model.layers):
    for t in sorted({s["tmpl"] for s in lay.specs}):
        ks = [k for k, s in enumerate(lay.specs) if s["tmpl"] == t]
        rows.append((f"layer {i}: {t} ×{len(ks)}", float(W[i, ks].sum())))
rows.sort(key=lambda r: r[1])
plt.figure(figsize=(6, 0.28 * len(rows) + 1))
plt.barh([r[0] for r in rows], [r[1] for r in rows])
for i, (_, v) in enumerate(rows):                    # the values: the biggest bar would hide the small ones
    plt.text(max(v, 0), i, f"  {v:+.3f}", va="center", fontsize=8)
plt.xlim(min(0, rows[0][1]) * 1.2, max(0.01, rows[-1][1]) * 1.25)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("loss rise when switched off (bits per token)")
plt.title("What each group of circuits is worth")
plt.show()

Groups at zero or below do nothing useful: those are the ones the search replaces. RG is at zero because there
is no library to read.

## 5. Agents learn to talk

The relay task from `experiments/swarm_relay.py` (results/README.md, section 1). A swarm of 2 agents shares one
message board:
- the **fact holder** reads 4 facts, `[FACT, key, value]`;
- the **asker** then reads questions, `[ASK, key, ?]`, and must predict each value.

Values are random for every example, so the asker can only beat chance (1 in 64) if the fact holder passes the facts
over the board. The fact holder gets no loss of its own: everything it learns to post comes from the asker's errors,
flowing back through the board.

This is the script's training loop for its `swarm` arm, written out here so you can see it. 300 steps take 1 to 4
minutes on a CPU.

In [ ]:
from exealm import ExeaLM, get_config
from experiments.swarm_relay import ASK, FACT, KEY0, N_KEYS, VAL0, make_batch, score

torch.manual_seed(0)
cfg = get_config("tiny", d=128, n_layers=4, slots=8, h=16, chunk=8, swarm_size=2, seed=0)
relay = ExeaLM(cfg).to(DEVICE)
opt = torch.optim.AdamW(relay.parameters(), lr=2e-3, betas=(0.9, 0.95), weight_decay=0.0)
rng, held_out_rng = np.random.default_rng(0), np.random.default_rng(10_000)
held_out = [make_batch(32, 4, cfg.chunk, held_out_rng) for _ in range(4)]  # 32 swarms of 2 agents, 4 facts each
curve, t0 = [], time.time()
for step in range(300):
    for g in opt.param_groups:
        g["lr"] = 2e-3 * min(1.0, (step + 1) / 200)                       # warm up
    x, y = make_batch(32, 4, cfg.chunk, rng)
    loss = relay(x.to(DEVICE), MemoryContext(swarm_size=2), y.to(DEVICE))[1]   # rows 0-1, 2-3, ... share a board
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(relay.parameters(), 1.0)
    opt.step()
    if (step + 1) % 25 == 0:
        shared = score(relay, held_out, 2, DEVICE)["acc"]                 # agents share the board
        cut = score(relay, held_out, 1, DEVICE)["acc"]                    # every agent alone
        curve.append((step + 1, shared, cut))
        print(f"step {step + 1:3d}   board shared {shared:6.1%}   board cut {cut:6.1%}   ({time.time() - t0:.0f} s)")

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot([c[0] for c in curve], [c[1] for c in curve], marker="o", markersize=4, label="board shared")
plt.plot([c[0] for c in curve], [c[2] for c in curve], marker="o", markersize=4, label="board cut")
plt.axhline(1 / N_KEYS, color="gray", linestyle="--", linewidth=1, label="chance (1 in 64)")
plt.ylim(0, 1)
plt.xlabel("training step")
plt.ylabel("asker's accuracy")
plt.title("The asker can only answer through the board")
plt.legend()
plt.show()

One swarm from the held-out batches, question by question. With the board cut, the asker can only guess.

In [ ]:
x, _ = held_out[0]
relay.eval()
with torch.no_grad():
    with_board = relay(x.to(DEVICE), MemoryContext(swarm_size=2))[0].argmax(-1).cpu()
    alone = relay(x.to(DEVICE), MemoryContext(swarm_size=1))[0].argmax(-1).cpu()


def value(tok):
    tok = int(tok)
    return tok - VAL0 if VAL0 <= tok < VAL0 + N_KEYS else f"token {tok}"


def entries(row, kind):
    """(position, key, value) of every FACT or ASK chunk in one agent's row."""
    return [(p, int(x[row, p + 1]) - KEY0, int(x[row, p + 2]) - VAL0)
            for p in range(0, x.shape[1], cfg.chunk) if int(x[row, p]) == kind]


print("The fact holder reads:", ", ".join(f"key {k} = {v}" for _, k, v in entries(0, FACT)))
print("The asker is asked:")
for p, k, v in entries(1, ASK):                     # the answer is predicted at the key's position
    print(f"  key {k:2d}?  answer {v:2d}   board shared says {value(with_board[1, p + 1])!s:>2}   "
          f"board cut says {value(alone[1, p + 1])}")

## What next

- Change something and run it again: more training steps, a different group of circuits switched off, or 4 agents in
  the relay (`python experiments/swarm_relay.py --agents 4`).
- Pick a task from the [to-do list](https://github.com/exea-labs/ExeaLM#to-do-list) or the
  [good first issues](https://github.com/exea-labs/ExeaLM/issues?q=is%3Aopen+label%3A%22good+first+issue%22).
- Read [CONTRIBUTING.md](https://github.com/exea-labs/ExeaLM/blob/main/CONTRIBUTING.md) before your first pull
  request.

In [ ]:
print(f"whole notebook: {(time.time() - START) / 60:.1f} minutes")